# Demo 03c — Logistic Regression

**Class 03 · Block 3** (slides: `slides/03_linear_models.md`)

Three live demos: the **spam** data (4601 e-mails, 57 features, 39% spam) and the **digits** data (1797 images, 10 classes).

* **L1 — Logistic regression:** the sigmoid, cross-entropy versus squared error, training with JAX, reading the weights.
* **L2 — Polynomial features and regularization** for a curved decision boundary.
* **L3 — More than two classes:** one class against the rest.

In [ ]:
import warnings
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.datasets import make_moons
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

jax.config.update("jax_enable_x64", True)
plt.rcParams["figure.figsize"] = (9, 4)
DATA = Path("../../datasets")

In [ ]:
spam = pl.read_csv(DATA / "spambase.csv.zst")
names = [c for c in spam.columns if c != "spam"]
X_raw = spam.select(names).to_numpy().astype(float)
y = spam["spam"].to_numpy().astype(int)
idx_tr, idx_te = (np.asarray(a) for a in train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=0))
X_tr_raw, X_te_raw, ytr, yte = X_raw[idx_tr], X_raw[idx_te], y[idx_tr], y[idx_te]
# Most frequencies are 0 and a few are huge: log(1 + x), then standardize with the training statistics.
scaler = StandardScaler().fit(np.log1p(X_tr_raw))
Xtr, Xte = np.asarray(scaler.transform(np.log1p(X_tr_raw))), np.asarray(scaler.transform(np.log1p(X_te_raw)))
print("train", Xtr.shape, "test", Xte.shape, "| spam fraction: train", ytr.mean().round(3), "test", yte.mean().round(3))
print("duplicate e-mails in the data:", int(spam.is_duplicated().sum()), "of", spam.height, "(a leak between train and test)")

## L1 — Logistic regression

The perceptron answers yes or no. **Logistic regression** answers with a probability:

$$p(y = 1 \mid x) = \sigma(w^\top x + b), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}$$

$z = w^\top x + b$ is the **log-odds**: $\log\frac{p}{1-p} = z$. Training maximizes the likelihood of the labels,
which is the same as minimizing the **binary cross-entropy**

$$\ell = -\big[y \log p + (1-y)\log(1-p)\big] = \log(1 + e^{z}) - y\,z .$$

**Why not the squared error?** Compare the two losses for a positive example ($y = 1$) as a function of the logit $z$.

In [ ]:
z = np.linspace(-6, 6, 300)
sig = 1 / (1 + np.exp(-z))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(z, sig)
axes[0].set(title=r"$\sigma(z)$", xlabel="z")
axes[1].plot(z, np.log1p(np.exp(-z)), label="cross-entropy")
axes[1].plot(z, (sig - 1) ** 2, label="squared error on p")
axes[1].set(title="loss for y = 1", xlabel="z")
axes[1].legend()
axes[2].plot(z, -(1 - sig), label="d cross-entropy / dz")
axes[2].plot(z, 2 * (sig - 1) * sig * (1 - sig), label="d squared error / dz")
axes[2].set(title="gradient for y = 1", xlabel="z")
axes[2].legend()
plt.tight_layout()
plt.show()

For a confidently **wrong** prediction ($z \ll 0$) the cross-entropy gradient is $-1$ (a strong push), while the
squared-error gradient goes to 0: the model would be stuck exactly when it is most wrong. The cross-entropy is also
**convex** in $(w, b)$; the squared error on a sigmoid is not.

The gradient of the mean cross-entropy is $\frac1N X^\top(p - y)$ (a residual again), but JAX gives it to us.

**A member of the GLM family** (slides: *Generalized Linear Models*). Linear regression (Gaussian noise, identity link) and logistic regression (Bernoulli, logit
link) are two members of the **generalized linear models**: a linear predictor $w^\top x + b$, a link function $g$, and a
distribution of $y$ from the exponential family. With the canonical link the gradient is always
$\frac1N X^\top(\hat\mu - y)$: features times residual. Poisson regression (counts, log link) is the third classic member
(`sklearn.linear_model.PoissonRegressor`).

In [ ]:
def bce(params, X_, y_, lam):
    z_ = X_ @ params["w"] + params["b"]
    return jnp.mean(jnp.logaddexp(0.0, z_) - y_ * z_) + 0.5 * lam * jnp.sum(params["w"] ** 2)   # log(1+e^z) - y z, + L2


bce_grad = jax.jit(jax.value_and_grad(bce))
lam = 1e-3
params = {"w": jnp.zeros(57), "b": jnp.zeros(())}
history = []
for _ in range(3000):
    value, g = bce_grad(params, Xtr, ytr.astype(float), lam)
    params = jax.tree.map(lambda p, gp: p - 0.5 * gp, params, g)
    history.append(float(value))

# scikit-learn's C multiplies the SUM of the losses: our lambda corresponds to C = 1 / (lambda * N).
sk = LogisticRegression(C=1 / (lam * len(ytr)), tol=1e-10, max_iter=10_000).fit(Xtr, ytr)
print("final training loss:", round(history[-1], 4))
print("max |weights ours - sklearn| =", float(np.abs(np.asarray(params["w"]) - sk.coef_[0]).max()))
proba = np.asarray(1 / (1 + np.exp(-(Xte @ params["w"] + params["b"]))))
print("max |test probability ours - sklearn| =", float(np.abs(proba - sk.predict_proba(Xte)[:, 1]).max()))
plt.plot(history)
plt.xlabel("gradient descent step")
plt.ylabel("training cross-entropy")
plt.show()

**Reading the weights.** The features are standardized, so $w_j$ is the change of the log-odds for one standard
deviation of feature $j$; $e^{w_j}$ is the **odds ratio**. We use the regularized version (`C = 0.1`) for stable,
interpretable weights.

In [ ]:
reg = LogisticRegression(C=0.1, max_iter=10_000).fit(Xtr, ytr)
table = pl.DataFrame({"feature": names, "weight": reg.coef_[0].round(3), "odds_ratio": np.exp(reg.coef_[0]).round(2)}).sort("weight")
print("most 'ham' features:\n", table.head(6))
print("most 'spam' features:\n", table.tail(6).reverse())

## L2 — Polynomial features and regularization for classification

A linear classifier draws a line (a hyperplane). With polynomial features of the inputs, the *same* model draws a curve,
and with too high a degree it draws a wiggly one that memorizes the noise. Regularization ($C = 1/(\lambda N)$ in scikit-learn: small
`C` is strong regularization) keeps it smooth.

In [ ]:
Xm, ym = make_moons(200, noise=0.3, random_state=0)
Xm_te, ym_te = make_moons(2000, noise=0.3, random_state=1)
gx, gy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
grid = np.c_[gx.ravel(), gy.ravel()]

degrees, Cs = (1, 3, 8, 15), (1e6, 0.1)
warnings.filterwarnings("ignore", category=ConvergenceWarning)          # the unregularized degree-15 fit may not converge
fig, axes = plt.subplots(2, 4, figsize=(14, 6.4), sharex=True, sharey=True)
for row, C in enumerate(Cs):
    for col, d in enumerate(degrees):
        poly = PolynomialFeatures(d, include_bias=False).fit(Xm)
        sc = StandardScaler().fit(poly.transform(Xm))
        model = LogisticRegression(C=C, max_iter=200_000, tol=1e-8).fit(sc.transform(poly.transform(Xm)), ym)
        zz = model.predict(sc.transform(poly.transform(grid))).reshape(gx.shape)
        tr_acc = model.score(sc.transform(poly.transform(Xm)), ym)
        te_acc = model.score(sc.transform(poly.transform(Xm_te)), ym_te)
        ax = axes[row, col]
        ax.contourf(gx, gy, zz, levels=[-0.5, 0.5, 1.5], colors=["#cfe3f5", "#fbdcc4"])
        ax.scatter(*Xm[ym == 0].T, s=8, color="tab:blue")
        ax.scatter(*Xm[ym == 1].T, s=8, color="tab:red")
        ax.set(title=f"degree {d}, C = {C:g}\ntrain {tr_acc:.2f}, test {te_acc:.3f}")
        print(f"degree {d:2d}, C = {C:<8g}: features {poly.n_output_features_:3d}  train acc {tr_acc:.3f}  test acc {te_acc:.3f}")
plt.tight_layout()
plt.show()

## L3 — More than two classes: one class against the rest

A logistic regression is binary. For $K$ classes we train $K$ binary models, *class $k$ against all the others*, and predict
the class with the largest probability. The handwritten digits (10 classes) are a small example. The alternative is the
**multinomial** logit (softmax): one joint model whose probabilities sum to 1.

In [ ]:
digits = pl.read_csv(DATA / "digits.csv.zst")
Xd, yd = digits.drop("target").to_numpy().astype(float) / 16.0, digits["target"].to_numpy()
tr_d, te_d = (np.asarray(a) for a in train_test_split(np.arange(len(yd)), test_size=0.25, stratify=yd, random_state=0))

ovr = OneVsRestClassifier(LogisticRegression(C=1.0, max_iter=5000)).fit(Xd[tr_d], yd[tr_d])
joint = LogisticRegression(C=1.0, max_iter=5000).fit(Xd[tr_d], yd[tr_d])        # multinomial (softmax), fitted jointly
print(f"one-vs-rest: {len(ovr.classes_)} binary models, test accuracy {ovr.score(Xd[te_d], yd[te_d]):.3f}")
print(f"multinomial: one joint model, test accuracy {joint.score(Xd[te_d], yd[te_d]):.3f}")
raw = 1 / (1 + np.exp(-np.asarray(ovr.decision_function(Xd[te_d]))))                  # one probability per binary model
print("sum of the K separate probabilities for 5 images:", raw.sum(axis=1)[:5].round(2), "(they need not sum to 1)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ConfusionMatrixDisplay.from_predictions(yd[te_d], ovr.predict(Xd[te_d]), cmap="Blues", colorbar=False, ax=axes[0])
axes[0].set_title("one-vs-rest: confusion matrix")
recall = [np.mean(ovr.predict(Xd[te_d])[yd[te_d] == k] == k) for k in range(10)]
k = int(np.argmin(recall))                                                              # the hardest class
binary = LogisticRegression(C=1.0, max_iter=5000).fit(Xd[tr_d], yd[tr_d] == k)           # class k against the rest
axes[1].imshow(binary.coef_[0].reshape(8, 8), cmap="coolwarm")
axes[1].set_title(f"weights of the binary model '{k} against the rest'")
plt.tight_layout()
plt.show()

## Summary

* Logistic regression = linear score + sigmoid, trained on the cross-entropy (convex): probabilities, not just labels; it is the Bernoulli GLM with the logit link.
* Non-linear boundaries need feature expansion; regularization keeps them smooth.
* More than two classes: one class against the rest (separate binary models, argmax), or the multinomial (softmax) model fitted jointly.
* Next: how to judge a classifier (`04_classification_metrics.ipynb`).